# NHS England Maternal Care Pathways Master Pipeline
## Stage 4 - Clean and recode MSDS variables

### Compiled by Ethan Phillips (Oxford)

### Last updated 08.12.2025

In [0]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql import SparkSession 
from pyspark.sql.functions import when, col, lit, count, max, last, first, min, avg, sum, collect_list, collect_set, countDistinct, to_date, datediff, array_contains, size, udf, format_number, regexp_replace, explode, array, array_union, desc_nulls_last
from pyspark.sql.types import IntegerType, StringType, NumericType, DateType, ArrayType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.ml.regression import GeneralizedLinearRegression as GLR

%run "/Workspace/Shared/Helper_Methods_EP"


In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
read_table_name = "msds_joined_filtered_filled_stage"

df_master = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{read_table_name}")

In [0]:
processed_features_to_keep = [
    "UniqPregID", "person_id_deid", "epikey", "fyear", "antenatal_appt_date",
    "last_period_date", "booking_org", "ccg_residence", "matserv_disch_date", "est_delivery_date", "ld_site_id", "ld_hosp_start_date", "ld_disch_date", "ld_disch_dest", "num_births", "labour_onset_date"
]

#####_Outcomes_

In [0]:
df_master = df_master.withColumns(
    {
        "neonatal_crit_incident": when(array_contains(col("neonatal_crit_incident"), "Y"), 1).when(size(col("neonatal_crit_incident"))==0, None).otherwise(0).cast("int"),
        "neonatal_critical_care": when(array_contains(col("neonatal_critical_care"), "Y"), 1).when(size(col("neonatal_critical_care"))==0, None).otherwise(0).cast('int'),
        "maternal_crit_incident": when(array_contains(col("maternal_crit_incident"), "Y"), 1).when(size(col("maternal_crit_incident"))==0, None).otherwise(0).cast('int'),
    }
)

processed_features_to_keep.extend(["neonatal_crit_incident", "neonatal_critical_care", "maternal_crit_incident"])

In [0]:
df_master = df_master.withColumns(
    {
        "very_low_birth_weight": when(col('birth_weight')<1500.0, 1).when(col('birth_weight').isNull(), None).otherwise(0),
        "low_birth_weight": when((col('birth_weight')>=1500.0) & (col('birth_weight')<2500.0), 1).when(col('birth_weight').isNull(), None).otherwise(0),
        "any_low_birth_weight": when(col("very_low_birth_weight") == 1, 1).when(col("low_birth_weight") == 1, 1).when(col("birth_weight").isNull(), None).otherwise(0),
        "high_birth_weight": when(col('birth_weight')>4000.0, 1).when(col('birth_weight').isNull(), None).otherwise(0),
        "normal_birth_weight": when((col('birth_weight')>=2500.0) & (col('birth_weight')<=4000.0), 1).when(col('birth_weight').isNull(), None).otherwise(0),
    }
)

processed_features_to_keep.extend(["very_low_birth_weight", "low_birth_weight", "any_low_birth_weight", "high_birth_weight", "normal_birth_weight"])

In [0]:
df_master = df_master.withColumns(
    {
        "normal_apgar": when(col('apgar_score')>=7, 1).when(col('apgar_score').isNull(), None).otherwise(0),
        "low_apgar": when(col('apgar_score')<7, 1).when(col('apgar_score').isNull(), None).otherwise(0),
        "severe_distress_apgar": when(col('apgar_score')<4, 1).when(col('apgar_score').isNull(), None).otherwise(0),
        "moderate_distress_apgar": when((col('apgar_score')<7) & (col('apgar_score')>=4), 1).when(col('apgar_score').isNull(), None).otherwise(0),
    }
)

processed_features_to_keep.extend(["normal_apgar", "low_apgar", "severe_distress_apgar", "moderate_distress_apgar"])

In [0]:
df_master = df_master.withColumns(
    {
        "outcome_live_birth": when(col('pregnancy_outcome')==1, 1).when(col('pregnancy_outcome').isNull(), None).otherwise(0),
        "outcome_antepartum_still_birth": when(col('pregnancy_outcome')==2, 1).when(col('pregnancy_outcome').isNull(), None).otherwise(0),
        "outcome_intrapartum_still_birth": when(col('pregnancy_outcome')==3, 1).when(col('pregnancy_outcome').isNull(), None).otherwise(0),
        "outcome_unknown_timing_still_birth": when(col('pregnancy_outcome')==4, 1).when(col('pregnancy_outcome').isNull(), None).otherwise(0),
        "outcome_any_still_birth": when((col('pregnancy_outcome')==2) | (col('pregnancy_outcome')==3) | (col('pregnancy_outcome')==4), 1).when(col('pregnancy_outcome').isNull(), None).otherwise(0),
        "outcome_termination_24w+": when(col('pregnancy_outcome')==5, 1).when(col('pregnancy_outcome').isNull(), None).otherwise(0),
        "outcome_other_not_listed": when(col('pregnancy_outcome')==98, 1).when(col('pregnancy_outcome').isNull(), None).otherwise(0)
    }
).withColumn(
    "pregnancy_outcome_missing", when((col('outcome_live_birth')==1) | (col('outcome_any_still_birth')==1) | (col('outcome_termination_24w+')==1) | (col('outcome_other_not_listed')==1), 0).when(col('pregnancy_outcome').isNull(), 1).otherwise(1)
)

processed_features_to_keep.extend(["outcome_live_birth", "outcome_antepartum_still_birth", "outcome_intrapartum_still_birth", "outcome_unknown_timing_still_birth", "outcome_any_still_birth"])

In [0]:
df_master = df_master.withColumns(
    {
        "extremely_preterm_birth": when(col('gestation_length_at_birth')<(28*7), 1).when(col('gestation_length_at_birth').isNull(), None).otherwise(0),
        "very_preterm_birth": when((col('gestation_length_at_birth')>=(28*7)) & (col('gestation_length_at_birth')<(32*7)), 1).when(col('gestation_length_at_birth').isNull(), None).otherwise(0),
        "preterm_birth_before_32w": when(col("extremely_preterm_birth") == 1, 1).when(col("very_preterm_birth") == 1, 1).when(col("gestation_length_at_birth").isNull(), None).otherwise(0),
        "moderate_to_late_preterm_birth": when((col('gestation_length_at_birth')>=(32*7)) & (col('gestation_length_at_birth')<(37*7)), 1).when(col('gestation_length_at_birth').isNull(), None).otherwise(0),
        "not_preterm_birth": when(col('gestation_length_at_birth')>=(37*7), 1).when(col('gestation_length_at_birth').isNull(), None).otherwise(0),
    }
)

processed_features_to_keep.extend(["extremely_preterm_birth", "very_preterm_birth", "moderate_to_late_preterm_birth", "not_preterm_birth", "preterm_birth_before_32w"])

In [0]:
df_master = df_master.withColumns(
    {
        #normal 
        'delivery_spont_vertex': when(col('delivery_method')==0, 1).when(col('delivery_method').isNull(), None).otherwise(0),

        #non-spontaneous cephalic
        'delivery_low_forcep_not_breech': when(col('delivery_method')==2, 1).when(col('delivery_method').isNull(), None).otherwise(0),
        'delivery_other_forcep_not_breech': when(col('delivery_method')==3, 1).when(col('delivery_method').isNull(), None).otherwise(0),
        'delivery_ventouse_vac': when(col('delivery_method')==4, 1).when(col('delivery_method').isNull(), None).otherwise(0),
        'delivery_other_ceph': when(col('delivery_method')==1, 1).when(col('delivery_method').isNull(), None).otherwise(0),

        #breech birth
        'delivery_breech': when(col('delivery_method')==5, 1).when(col('delivery_method').isNull(), None).otherwise(0),
        'delivery_breech_extract': when(col('delivery_method')==6, 1).when(col('delivery_method').isNull(), None).otherwise(0),

        #any csection
        'delivery_elect_csection': when(col('delivery_method')==7, 1).when(col('delivery_method').isNull(), None).otherwise(0),
        'delivery_emergency_csection': when(col('delivery_method')==8, 1).when(col('delivery_method').isNull(), None).otherwise(0),

        #code to null 
        'delivery_not_listed': when(col('delivery_method')==9, 1).when(col('delivery_method').isNull(), None).otherwise(0),
    }
).withColumns({
    "delivery_any_csection": when(col("delivery_elect_csection")==1, 1).when(col("delivery_emergency_csection")==1, 1).when(col('delivery_method').isNull() | (col("delivery_not_listed")==1), None).otherwise(0),
    "delivery_standard": when(col("delivery_spont_vertex")==1,1).when(col('delivery_method').isNull() | (col("delivery_not_listed")==1), None).otherwise(0),
    "delivery_nonstandard_ceph": when(col("delivery_low_forcep_not_breech")==1,1).when(col("delivery_other_forcep_not_breech")==1,1).when(col("delivery_ventouse_vac")==1,1).when(col("delivery_other_ceph")==1,1).when(col('delivery_method').isNull() | (col("delivery_not_listed")==1), None).otherwise(0),
    "delivery_any_breech": when(col("delivery_breech")==1, 1).when(col("delivery_breech_extract")==1, 1).when(col('delivery_method').isNull() | (col("delivery_not_listed")==1), None).otherwise(0),
})
# add non-ceph non-csection

processed_features_to_keep.extend([
    "delivery_spont_vertex",
    "delivery_other_ceph",
    "delivery_low_forcep_not_breech",
    "delivery_other_forcep_not_breech",
    "delivery_ventouse_vac",
    "delivery_breech",
    "delivery_breech_extract",
    "delivery_elect_csection",
    "delivery_emergency_csection",
    "delivery_not_listed",
    "delivery_any_csection",
    "delivery_standard",
    "delivery_nonstandard_ceph",
    "delivery_any_breech",
])


In [0]:
df_master = df_master.withColumns(
  {
    'fetus_presentation_cephalic': when(col('fetus_presentation')==1, 1).when(col('fetus_presentation').isNull(), None).otherwise(0),
    'fetus_presentation_breech': when(col('fetus_presentation')==2, 1).when(col('fetus_presentation').isNull(), None).otherwise(0),
    'fetus_presentation_transverse': when(col('fetus_presentation')==3, 1).when(col('fetus_presentation').isNull(), None).otherwise(0),
  }
).withColumn('fetus_presentation_missing', when(col('fetus_presentation').isNull(), 1).when((col('fetus_presentation_cephalic')==0) & (col('fetus_presentation_breech')==0) & (col('fetus_presentation_transverse')==0), 1).otherwise(0))

processed_features_to_keep.extend([
    "fetus_presentation_cephalic",
    "fetus_presentation_breech",
    "fetus_presentation_transverse"
])

#####_Explanatory Variables_

In [0]:
df_master = df_master.filter(col("age_at_booking") >= 13).filter(col("age_at_booking") <= 60)

processed_features_to_keep.extend(["age_at_booking"])

In [0]:
df_master = df_master.withColumns({
    "booking_age_u20": when((col("age_at_booking").isNotNull()) & (col("age_at_booking")<20), 1).when(col("age_at_booking").isNull(), None).otherwise(0),
    "booking_age_20_29": when((col("age_at_booking").isNotNull()) & (col("age_at_booking")>=20) & (col("age_at_booking")<30), 1).when(col("age_at_booking").isNull(), None).otherwise(0),
    "booking_age_30_39": when((col("age_at_booking").isNotNull()) & (col("age_at_booking")>=30) & (col("age_at_booking")<40), 1).when(col("age_at_booking").isNull(), None).otherwise(0),
    "booking_age_40_49": when((col("age_at_booking").isNotNull()) & (col("age_at_booking")>=40) & (col("age_at_booking")<50), 1).when(col("age_at_booking").isNull(), None).otherwise(0),
    "booking_age_o50": when((col("age_at_booking").isNotNull()) & (col("age_at_booking")>=50), 1).when(col("age_at_booking").isNull(), None).otherwise(0),
    # New age bands
    "booking_age_u35": when((col("age_at_booking").isNotNull()) & (col("age_at_booking")<35), 1).when(col("age_at_booking").isNull(), None).otherwise(0),
    "booking_age_35_40": when((col("age_at_booking").isNotNull()) & (col("age_at_booking")>=35) & (col("age_at_booking")<40), 1).when(col("age_at_booking").isNull(), None).otherwise(0),
    "booking_age_o40": when((col("age_at_booking").isNotNull()) & (col("age_at_booking")>=40), 1).when(col("age_at_booking").isNull(), None).otherwise(0),
})

processed_features_to_keep.extend(["booking_age_u20", "booking_age_20_29", "booking_age_30_39", "booking_age_40_49", "booking_age_o50", "booking_age_u35", "booking_age_35_40", "booking_age_o40"])

In [0]:
df_master = df_master.withColumns({
    "mother_first_weight": when(col("mother_first_weight") > 200, None).when(col("mother_first_weight") < 35, None).otherwise(col("mother_first_weight")),
    "mother_last_weight": when(col("mother_last_weight") > 200, None).when(col("mother_last_weight") < 35, None).otherwise(col("mother_last_weight")),
    "mother_avg_weight": when(col("mother_avg_weight") > 200, None).when(col("mother_avg_weight") < 35, None).otherwise(col("mother_avg_weight")),
}).withColumn(
    "mother_weight_change", col("mother_last_weight") - col("mother_first_weight")
)

processed_features_to_keep.extend(["mother_first_weight", "mother_last_weight", "mother_weight_change", "mother_avg_weight"])

print(f"Percent missingness in weight: {100*df_master.filter(col("mother_avg_weight").isNull()).count()/df_master.count():.2f}%")

In [0]:
df_master = df_master.withColumns({
    "mother_low_weight": when(col("mother_avg_weight").isNotNull() & (col("mother_avg_weight")<=60.0), 1).when(col("mother_avg_weight").isNotNull() & (col("mother_avg_weight")>60.0), 0).otherwise(None),
    "mother_med_weight": when(col("mother_avg_weight").isNotNull() & (col("mother_avg_weight")>60.0) & (col("mother_avg_weight")<=83.0), 1).when(col("mother_avg_weight").isNotNull() & ((col("mother_avg_weight")<=60.0) | (col("mother_avg_weight")>83.0)), 0).otherwise(None),
    "mother_high_weight": when(col("mother_avg_weight").isNotNull() & (col("mother_avg_weight")>83.0), 1).when(col("mother_avg_weight").isNotNull() & (col("mother_avg_weight")<=83.0), 0).otherwise(None)
})

# Categorise based on average height and BMI cut offs
# Average woman's height in the UK: 1.62m -> squared: 2.6244
df_master = df_master.withColumn("estimated_bmi", F.round((col("mother_avg_weight") / (lit(2.6244))), 1))

df_master = df_master.withColumns({
    "mother_bmi_u185":  when(col("estimated_bmi").isNotNull() & (col("estimated_bmi")<=18.5), 1).when(col("estimated_bmi").isNotNull() & (col("estimated_bmi")>18.5), 0).otherwise(None),
    "mother_bmi_185_249":  when(col("estimated_bmi").isNotNull() & (col("estimated_bmi")>18.5) & (col("estimated_bmi")<=24.9), 1).when(col("estimated_bmi").isNotNull() & ((col("estimated_bmi")<=18.5) | (col("estimated_bmi")>24.9)), 0).otherwise(None),
    "mother_bmi_250_299":  when(col("estimated_bmi").isNotNull() & (col("estimated_bmi")>25.0) & (col("estimated_bmi")<=29.9), 1).when(col("estimated_bmi").isNotNull() & ((col("estimated_bmi")>25.0) | (col("estimated_bmi")<=29.9)), 0).otherwise(None),
    "mother_bmi_o300":  when(col("estimated_bmi").isNotNull() & (col("estimated_bmi")>30.0), 1).when(col("estimated_bmi").isNotNull() & (col("estimated_bmi")<=30.0), 0).otherwise(None)
})

processed_features_to_keep.extend([
    "mother_low_weight", "mother_med_weight", "mother_high_weight",
    "mother_bmi_u185", "mother_bmi_185_249", "mother_bmi_250_299", "mother_bmi_o300"
])

In [0]:
df_master = (
    df_master
    .withColumn("mother_avg_height", when((col("mother_avg_height") > 130.0) & (col("mother_avg_height") < 210.0), col("mother_avg_height")/100).otherwise(col("mother_avg_height")))
    .withColumn("mother_avg_height", when((col("mother_avg_height") < 1.3), None).when(col("mother_avg_height") > 2.1, None).otherwise(col("mother_avg_height")))
)

print(f"Percent missingness in height: {100*df_master.filter(col("mother_avg_height").isNull()).count()/df_master.count():.2f}%")

In [0]:
df_master = df_master.withColumns({
    "mother_bmi": when(col("mother_avg_height").isNotNull() & col("mother_avg_weight").isNotNull(), col("mother_avg_weight")/(col("mother_avg_height")**2)).otherwise(None)
})

print(f"Percent missingness in calculated BMI: {100*df_master.filter(col("mother_bmi").isNotNull()).count()/df_master.count():.2f}%")

In [0]:
df_master = df_master.withColumns(
    {
        "ever_smoking": when(col('max_cigs_per_day')>0, 1).otherwise(0), # assume missing smoking data = non-smoker
        "ever_drinking": when(col('max_alcohol_per_week')>0, 1).when(col('max_alcohol_per_week').isNull(), None).otherwise(0),
    }
).withColumns(
    {
        "stopped_smoking": when((col('first_cigs_per_day')>0) & (col('last_cigs_per_day')==0), 1).when(col('ever_smoking')==0, None).otherwise(0),
        "stopped_drinking": when((col('first_alcohol_per_week')>0) & (col('last_alcohol_per_week')==0), 1).when(col('ever_drinking')==0, None).when(col('last_alcohol_per_week').isNull(), None).otherwise(0),
        "ever_substance_use": when((col("ever_smoking").isNotNull() & (col("ever_smoking") == 1)) | (col("ever_drinking").isNotNull() & (col("ever_drinking") == 1)), 1).when((col("ever_smoking").isNotNull() & (col("ever_smoking") == 0)) | (col("ever_drinking").isNotNull() & (col("ever_drinking") == 0)), 0).otherwise(None)
    }
)

processed_features_to_keep.extend(["ever_smoking", "ever_drinking", "stopped_smoking", "stopped_drinking", "ever_substance_use"])

In [0]:
df_master = df_master.withColumn(
    "abnormal_ultrasound_ind", 
    when(array_contains(col("abnormal_ultrasound"), "Y"), 1).when(size(col("abnormal_ultrasound"))==0, None).otherwise(0).cast('int')
)
                                                               
df_master = df_master.withColumn(
    "social_factors_ind", 
    when(array_contains(col("social_factors"), "Y"), 1).when(size(col("social_factors"))==0, None).otherwise(0).cast('int')
)

df_master = df_master.withColumn(
    "disability_ind", 
    when(array_contains(col("disability"), "Y"), 1).when(size(col("disability"))==0, None).otherwise(0).cast('int')
)

df_master = df_master.withColumn(
    "support_ind", 
    when(array_contains(col("support_status"), "Y"), 1).when(array_contains(col("support_status"), "N"), 0).otherwise(None).cast('int')
)

processed_features_to_keep.extend(["abnormal_ultrasound_ind", "social_factors_ind", "disability_ind", "support_ind"])

In [0]:
df_master = df_master.withColumn(
    'folic_acid_pre_pregnancy', 
    when(array_contains(col('folic_acid'), '01'), 1).otherwise(0)
).withColumn(
    'folic_acid_while_pregnant',
    when(col('folic_acid_pre_pregnancy')==1, 1).when(array_contains(col('folic_acid'), '02'), 1).otherwise(0)
).withColumn(
    'no_folic_acid',
    when(array_contains(col('folic_acid'), '03'), 1).when(col('folic_acid_while_pregnant')==1, 0).otherwise(0)
).withColumn(
    'folic_acid_missing',
    when(size(col('folic_acid'))==0, 1).when(col('folic_acid_while_pregnant')==1, 0).when(col('no_folic_acid')==1, 0).otherwise(1)
).withColumns(
    {
        'folic_acid_pre_pregnancy': when(col('folic_acid_missing')==1, None).otherwise(col('folic_acid_pre_pregnancy')),
        'folic_acid_while_pregnant': when(col('folic_acid_missing')==1, None).otherwise(col('folic_acid_while_pregnant')),
        'no_folic_acid': when(col('folic_acid_missing')==1, None).otherwise(col('no_folic_acid')),
    }
)

processed_features_to_keep.extend(["folic_acid_pre_pregnancy", "folic_acid_while_pregnant", "no_folic_acid"])

In [0]:
df_master = df_master.withColumns(
    {
        'ethnicity_white_british': when(array_contains(col("ethnicity"), 'A'), 1).otherwise(0),
        'ethnicity_white_irish': when(array_contains(col("ethnicity"), 'B'), 1).otherwise(0),
        'ethnicity_white_other': when(array_contains(col("ethnicity"), 'C'), 1).otherwise(0),
        'ethnicity_mixed_white_black': when(array_contains(col("ethnicity"), 'D'), 1).when(array_contains(col("ethnicity"), 'E'), 1).otherwise(0),
        'ethnicity_mixed_white_asian': when(array_contains(col("ethnicity"), 'F'), 1).otherwise(0),
        'ethnicity_mixed_other': when(array_contains(col("ethnicity"), 'G'), 1).otherwise(0),
        'ethnicity_asian_indian': when(array_contains(col("ethnicity"), 'H'), 1).otherwise(0),
        'ethnicity_asian_pakistani': when(array_contains(col("ethnicity"), 'J'), 1).otherwise(0),
        'ethnicity_asian_bangladeshi': when(array_contains(col("ethnicity"), 'K'), 1).otherwise(0),
        'ethnicity_asian_other': when(array_contains(col("ethnicity"), 'L'), 1).otherwise(0),
        'ethnicity_black_caribbean': when(array_contains(col("ethnicity"), 'M'), 1).otherwise(0),
        'ethnicity_black_african': when(array_contains(col("ethnicity"), 'N'), 1).otherwise(0),
        'ethnicity_black_other': when(array_contains(col("ethnicity"), 'P'), 1).otherwise(0),
        'ethnicity_asian_chinese': when(array_contains(col("ethnicity"), 'R'), 1).otherwise(0),
        'ethnicity_other': when(array_contains(col("ethnicity"), 'S'), 1).otherwise(0),
        'ethnicity_not_stated': when(array_contains(col("ethnicity"), 'Z'), 1).when(array_contains(col("ethnicity"), '99'), 1).otherwise(0),
        'ethnicity_missing': when(size(col("ethnicity")) == 0, 1).otherwise(0)
    }
).withColumns(
    {
        'ethnicity_white': when(col('ethnicity_white_british')==1, 1).when(col('ethnicity_white_irish')==1, 1).when(col('ethnicity_white_other')==1, 1).otherwise(0),
        'ethnicity_mixed': when(col('ethnicity_mixed_white_black')==1, 1).when(col('ethnicity_mixed_white_asian')==1, 1).when(col('ethnicity_mixed_other')==1, 1).otherwise(0),
        'ethnicity_asian': when(col('ethnicity_asian_indian')==1, 1).when(col('ethnicity_asian_pakistani')==1, 1).when(col('ethnicity_asian_bangladeshi')==1, 1).when(col('ethnicity_asian_other')==1, 1).when(col('ethnicity_asian_chinese')==1, 1).otherwise(0),
        'ethnicity_black': when(col('ethnicity_black_caribbean')==1, 1).when(col('ethnicity_black_african')==1, 1).when(col('ethnicity_black_other')==1, 1).otherwise(0)
    }
).withColumn(
    'ethnicity_missing', 
    when(col('ethnicity_missing')==1, 1).when((col('ethnicity_white') + col('ethnicity_mixed') + col('ethnicity_asian') + col('ethnicity_black') + col('ethnicity_other'))==0, 1).otherwise(0)
).withColumns(
    {
        'ethnic_white_reg': when(col('ethnicity_white')==1, 1).when(col('ethnicity_missing')==1, None).otherwise(0),
        'ethnic_black_caribbean_reg': when(col('ethnicity_black_caribbean')==1, 1).when(col('ethnicity_missing')==1, None).otherwise(0),
        'ethnic_black_african_reg': when(col('ethnicity_black_african')==1, 1).when(col('ethnicity_missing')==1, None).otherwise(0),
        'ethnic_asian_indian_reg': when(col('ethnicity_asian_indian')==1, 1).when(col('ethnicity_missing')==1, None).otherwise(0),
        'ethnic_asian_pakistani_reg': when(col('ethnicity_asian_pakistani')==1, 1).when(col('ethnicity_missing')==1, None).otherwise(0),
        'ethnic_asian_bangladeshi_reg': when(col('ethnicity_asian_bangladeshi')==1, 1).when(col('ethnicity_missing')==1, None).otherwise(0),
        'ethnic_mixed_reg': when(col('ethnicity_mixed')==1, 1).when(col('ethnicity_missing')==1, None).otherwise(0),
        'ethnic_other_reg': when((col('ethnic_white_reg') + col('ethnic_black_caribbean_reg') + col('ethnic_black_african_reg') + col('ethnic_asian_indian_reg') + col('ethnic_asian_pakistani_reg') + col('ethnic_asian_bangladeshi_reg') + col('ethnic_mixed_reg'))==0, 1).when(col('ethnicity_missing')==1, None).otherwise(0),
    }
).withColumns({
    "ethnic_black_reg": when(col('ethnic_black_caribbean_reg')==1, 1).when(col('ethnic_black_african_reg')==1, 1).when(col('ethnicity_missing')==1, None).otherwise(0),
    "ethnic_south_asian_reg": when(col('ethnic_asian_indian_reg')==1, 1).when(col('ethnic_asian_pakistani_reg')==1, 1).when(col('ethnic_asian_bangladeshi_reg')==1, 1).when(col('ethnicity_missing')==1, None).otherwise(0),
})

processed_features_to_keep.extend(["ethnic_white_reg", "ethnic_black_caribbean_reg", "ethnic_black_african_reg", "ethnic_black_reg", "ethnic_asian_indian_reg", "ethnic_asian_pakistani_reg", "ethnic_asian_bangladeshi_reg", "ethnic_south_asian_reg", "ethnic_mixed_reg", "ethnic_other_reg"])

In [0]:
df_master = df_master.withColumns(
    {
        'baby_ethnicity_white_british': when(array_contains(col("baby_ethnicity"), 'A'), 1).otherwise(0),
        'baby_ethnicity_white_irish': when(array_contains(col("baby_ethnicity"), 'B'), 1).otherwise(0),
        'baby_ethnicity_white_other': when(array_contains(col("baby_ethnicity"), 'C'), 1).otherwise(0),
        'baby_ethnicity_mixed_white_black': when(array_contains(col("baby_ethnicity"), 'D'), 1).when(array_contains(col("baby_ethnicity"), 'E'), 1).otherwise(0),
        'baby_ethnicity_mixed_white_asian': when(array_contains(col("baby_ethnicity"), 'F'), 1).otherwise(0),
        'baby_ethnicity_mixed_other': when(array_contains(col("baby_ethnicity"), 'G'), 1).otherwise(0),
        'baby_ethnicity_asian_indian': when(array_contains(col("baby_ethnicity"), 'H'), 1).otherwise(0),
        'baby_ethnicity_asian_pakistani': when(array_contains(col("baby_ethnicity"), 'J'), 1).otherwise(0),
        'baby_ethnicity_asian_bangladeshi': when(array_contains(col("baby_ethnicity"), 'K'), 1).otherwise(0),
        'baby_ethnicity_asian_other': when(array_contains(col("baby_ethnicity"), 'L'), 1).otherwise(0),
        'baby_ethnicity_black_caribbean': when(array_contains(col("baby_ethnicity"), 'M'), 1).otherwise(0),
        'baby_ethnicity_black_african': when(array_contains(col("baby_ethnicity"), 'N'), 1).otherwise(0),
        'baby_ethnicity_black_other': when(array_contains(col("baby_ethnicity"), 'P'), 1).otherwise(0),
        'baby_ethnicity_asian_chinese': when(array_contains(col("baby_ethnicity"), 'R'), 1).otherwise(0),
        'baby_ethnicity_other': when(array_contains(col("baby_ethnicity"), 'S'), 1).otherwise(0),
        'baby_ethnicity_not_stated': when(array_contains(col("baby_ethnicity"), 'Z'), 1).when(array_contains(col("baby_ethnicity"), '99'), 1).otherwise(0),
        'baby_ethnicity_missing': when(size(col("baby_ethnicity")) == 0, 1).otherwise(0)
    }
).withColumns(
    {
        'baby_ethnicity_white': when(col('baby_ethnicity_white_british')==1, 1).when(col('baby_ethnicity_white_irish')==1, 1).when(col('baby_ethnicity_white_other')==1, 1).otherwise(0),
        'baby_ethnicity_mixed': when(col('baby_ethnicity_mixed_white_black')==1, 1).when(col('baby_ethnicity_mixed_white_asian')==1, 1).when(col('baby_ethnicity_mixed_other')==1, 1).otherwise(0),
        'baby_ethnicity_asian': when(col('baby_ethnicity_asian_indian')==1, 1).when(col('baby_ethnicity_asian_pakistani')==1, 1).when(col('baby_ethnicity_asian_bangladeshi')==1, 1).when(col('baby_ethnicity_asian_other')==1, 1).when(col('baby_ethnicity_asian_chinese')==1, 1).otherwise(0),
        'baby_ethnicity_black': when(col('baby_ethnicity_black_caribbean')==1, 1).when(col('baby_ethnicity_black_african')==1, 1).when(col('baby_ethnicity_black_other')==1, 1).otherwise(0)
    }
).withColumn('baby_ethnicity_missing', when(col('baby_ethnicity_missing')==1, 1).when((col('baby_ethnicity_white') + col('baby_ethnicity_mixed') + col('baby_ethnicity_asian') + col('baby_ethnicity_black') + col('baby_ethnicity_other'))==0, 1).otherwise(0))

In [0]:
df_master = df_master.withColumns(
    {
        'mother_employ_transition': when(col('mother_employ_first')!=col('mother_employ_last'), 1).when(col('mother_employ_first').isNull(), None).otherwise(0),
        'partner_employ_transition': when(col('partner_employ_first')!=col('partner_employ_last'), 1).when(col('partner_employ_first').isNull(), None).otherwise(0),
    }
).withColumns(
    {
        'mother_employed': when((col('mother_employ_transition')==0) & (col('mother_employ_first')==1), 1).when(col('mother_employ_first').isNull(), None).otherwise(0),
        'mother_unemployed': when((col('mother_employ_transition')==0) & (col('mother_employ_first')==2), 1).when(col('mother_employ_first').isNull(), None).otherwise(0),
        'mother_student': when((col('mother_employ_transition')==0) & (col('mother_employ_first')==3), 1).when(col('mother_employ_first').isNull(), None).otherwise(0),
        'mother_sick_or_disabled_benefits': when((col('mother_employ_transition')==0) & (col('mother_employ_first')==4), 1).when(col('mother_employ_first').isNull(), None).otherwise(0),
        'mother_unpaid_work': when((col('mother_employ_transition')==0) & (col('mother_employ_first')==7), 1).when(col('mother_employ_first').isNull(), None).otherwise(0),
        'mother_not_in_labour_market': when((col('mother_employ_transition')==0) & ((col('mother_employ_first')==5) | (col('mother_employ_first')==6) | (col('mother_employ_first')==8)), 1).when(col('mother_employ_first').isNull(), None).otherwise(0),
        'mother_transition_to_work': when((col('mother_employ_transition')==1) & (col('mother_employ_last')==1), 1).when(col('mother_employ_first').isNull(), None).otherwise(0),
        'mother_transition_out_of_work': when((col('mother_employ_transition')==1) & (col('mother_employ_first')==1), 1).when(col('mother_employ_first').isNull(), None).otherwise(0),
    }
).withColumns(
    {
        'partner_employed': when((col('partner_employ_transition')==0) & (col('partner_employ_first')==1), 1).when(col('partner_employ_first').isNull(), None).otherwise(0),
        'partner_unemployed': when((col('partner_employ_transition')==0) & (col('partner_employ_first')==2), 1).when(col('partner_employ_first').isNull(), None).otherwise(0),
        'partner_student': when((col('partner_employ_transition')==0) & (col('partner_employ_first')==3), 1).when(col('partner_employ_first').isNull(), None).otherwise(0),
        'partner_sick_or_disabled_benefits': when((col('partner_employ_transition')==0) & (col('partner_employ_first')==4), 1).when(col('partner_employ_first').isNull(), None).otherwise(0),
        'partner_unpaid_work': when((col('partner_employ_transition')==0) & (col('partner_employ_first')==7), 1).when(col('partner_employ_first').isNull(), None).otherwise(0),
        'partner_not_in_labour_market': when((col('partner_employ_transition')==0) & ((col('partner_employ_first')==5) | (col('partner_employ_first')==6) | (col('partner_employ_first')==8)), 1).when(col('partner_employ_first').isNull(), None).otherwise(0),
        'partner_transition_to_work': when((col('partner_employ_transition')==1) & (col('partner_employ_last')==1), 1).when(col('partner_employ_first').isNull(), None).otherwise(0),
        'partner_transition_out_of_work': when((col('partner_employ_transition')==1) & (col('partner_employ_first')==1), 1).when(col('partner_employ_first').isNull(), None).otherwise(0),
    }
)

processed_features_to_keep.extend([
    "mother_employed",
    "mother_unemployed",
    "mother_student",
    "mother_sick_or_disabled_benefits",
    "mother_unpaid_work",
    "mother_not_in_labour_market",
    "mother_transition_to_work",
    "mother_transition_out_of_work",
    "partner_employed",
    "partner_unemployed",
    "partner_student",
    "partner_sick_or_disabled_benefits",
    "partner_unpaid_work",
    "partner_not_in_labour_market",
    "partner_transition_to_work",
    "partner_transition_out_of_work"
])

In [0]:
df_master = df_master.withColumns(
    {
        'lang_english': when(array_contains(col("language"), 'en'), 1).otherwise(0),
        'lang_missing': when((size(col("language")) == 0) | array_contains(col("language"), 'xx'), 1).otherwise(0),
        'lang_french': when(array_contains(col("language"), 'fr'), 1).otherwise(0),
        'lang_german': when(array_contains(col("language"), 'de'), 1).otherwise(0),
        'lang_italian': when(array_contains(col("language"), 'it'), 1).otherwise(0),
        'lang_welsh': when(array_contains(col("language"), 'cy'), 1).otherwise(0),
        'lang_irish': when(array_contains(col("language"), 'ga'), 1).otherwise(0),
        'lang_chinese': when(array_contains(col("language"), 'zh'), 1).otherwise(0),
        'lang_arabic': when(array_contains(col("language"), 'ar'), 1).otherwise(0),
        'lang_polish': when(array_contains(col("language"), 'pl'), 1).otherwise(0),
        'lang_punjabi': when(array_contains(col("language"), 'pa'), 1).otherwise(0),
        'lang_romanian': when(array_contains(col("language"), 'ro'), 1).otherwise(0),
        'lang_spanish': when(array_contains(col("language"), 'es'), 1).otherwise(0),
        'lang_portuguese': when(array_contains(col("language"), 'pt'), 1).otherwise(0),
        'lang_urdu': when(array_contains(col("language"), 'ur'), 1).otherwise(0),
        'lang_gujarati': when(array_contains(col("language"), 'gu'), 1).otherwise(0),
        'lang_bengali': when(array_contains(col("language"), 'bn'), 1).otherwise(0)
    }
)

df_master = df_master.withColumns({
        'lang_missing': when(col("lang_missing") == 1, 1).when((col("lang_english") == 1) | (col("lang_french") == 1) | (col("lang_german") == 1) | (col("lang_irish") == 1) | (col("lang_welsh") == 1) | (col("lang_italian") == 1) | (col("lang_chinese") == 1) | (col("lang_arabic") == 1) | (col("lang_polish") == 1) | (col("lang_punjabi") == 1) | (col("lang_bengali") == 1) | (col("lang_urdu") == 1) | (col("lang_gujarati") == 1) | (col("lang_spanish") == 1) | (col("lang_romanian") == 1) | (col("lang_portuguese") == 1), 0).otherwise(1),
}).withColumns({
    "lang_not_english": when(col("lang_english") == 1, 0).when(col("lang_missing") == 1, None).otherwise(1),
    "lang_english": when(col("lang_english") == 1, 1).when(col("lang_missing") == 1, None).otherwise(0),
})

processed_features_to_keep.extend(["lang_english", "lang_not_english"])



In [0]:
df_master = df_master.withColumns(
    {
        'is_religious': when(array_contains(col('religion'), 62458008), 1).otherwise(0),
        'religion_hindu': when(array_contains(col('religion'), 160545000), 1).otherwise(0),
        'religion_declined': when(array_contains(col('religion'), 160545000), 1).otherwise(0),
        'religion_christian': when(array_contains(col('religion'), 160549006), 1).otherwise(0),
        'religion_agnostic': when(array_contains(col('religion'), 160567004), 1).otherwise(0),
        'religion_jewish': when(array_contains(col('religion'), 160543007), 1).otherwise(0),
        'religion_sikh': when(array_contains(col('religion'), 366740002), 1).otherwise(0),
        'religion_muslim': when(array_contains(col('religion'), 309884000), 1).otherwise(0),
        'religion_buddhist': when(array_contains(col('religion'), 309687009), 1).otherwise(0),
        'religion_unknown': when(array_contains(col('religion'), 205081000000105), 1).otherwise(0),
        'religion_pagan': when(array_contains(col('religion'), 205141000000101), 1).otherwise(0),
        'religion_jain': when(array_contains(col('religion'), 429787006), 1).otherwise(0),
        'religion_bahai': when(array_contains(col('religion'), 429732005), 1).otherwise(0),
        'religion_zoroastrian': when(array_contains(col('religion'), 429790000), 1).otherwise(0),
        'religion_missing': when(size(col('religion'))==0, 1).otherwise(0),
    }
).withColumn('not_religious', when(array_contains(col('religion'), 160552003), 1).when(col('religion_agnostic')==1, 1).otherwise(0)).withColumns(
    {
        'is_religious': when((col('is_religious')==1) | (col('religion_hindu')==1) | (col('religion_christian')==1) | (col('religion_jewish')==1) | (col('religion_sikh')==1) | (col('religion_muslim')==1) | (col('religion_buddhist')==1) | (col('religion_pagan')==1) | (col('religion_jain')==1) | (col('religion_bahai')==1) | (col('religion_zoroastrian')==1), 1).otherwise(0),
    }
).withColumn(
    'religion_missing', 
    when((col('religion_missing')==1) | (col('religion_unknown')==1), 1).when((col('is_religious')==1) | (col('not_religious')==1), 0).otherwise(None)
).withColumns(
    {
        'is_religious': when(col('is_religious')==1, 1).when(col('not_religious')==1, 0).otherwise(None),
        'not_religious': when(col('not_religious')==1, 1).when(col('is_religious')==1, 0).otherwise(None)
    }
)

processed_features_to_keep.extend(["is_religious", "not_religious"])

In [0]:
df_master = df_master.withColumns(
    {
        'imd_missing': when(col('imd_decile').isNull(), 1).otherwise(0),
        'deprived_reg': when(col('imd_decile').isNull(), None).when(col('imd_decile') <= 3, 1).otherwise(0)
    }
)

processed_features_to_keep.extend(["imd_decile", "deprived_reg"])

In [0]:
df_master = df_master.withColumns(
    {
        'sexual_orientation_female_homosexual': when(array_contains(col('sexual_orientation'), 89217008), 1).otherwise(0),
        'sexual_orientation_unknown': when(array_contains(col('sexual_orientation'), 699042003), 1).when(array_contains(col('sexual_orientation'), 440583007), 1).otherwise(0),
        'sexual_orientation_bisexual': when(array_contains(col('sexual_orientation'), 42035005), 1).otherwise(0),
        'sexual_orientation_heterosexual': when(array_contains(col('sexual_orientation'), 20430005), 1).otherwise(0),
        'sexual_orientation_undecided': when(array_contains(col('sexual_orientation'), 1064711000000108), 1).otherwise(0),
        'sexual_orientation_asexual': when(array_contains(col('sexual_orientation'), 765288000), 1).otherwise(0),
        'sexual_orientation_male_homosexual': when(array_contains(col('sexual_orientation'), 76102007), 1).otherwise(0),
        'sexual_orientation_missing': when(size(col('sexual_orientation'))==0,1).otherwise(0)
    }
).withColumns(
    {
        'sexual_orientation_homosexual_or_bisexual': when((col('sexual_orientation_female_homosexual')==1) | (col('sexual_orientation_male_homosexual')==1) | (col('sexual_orientation_bisexual')==1), 1).when(col('sexual_orientation_missing')==1, None).otherwise(0),
        'sexual_orientation_heterosexual': when(col('sexual_orientation_heterosexual')==1, 1).when(col('sexual_orientation_missing')==1, None).otherwise(0),
        'sexual_orientation_unknown': when(col('sexual_orientation_unknown')==1, 1).when(col('sexual_orientation_missing')==1, None).otherwise(0),
    }
).withColumn('sexual_orientation_other', when((col('sexual_orientation_heterosexual')==1) | (col('sexual_orientation_homosexual_or_bisexual')==1) | (col('sexual_orientation_unknown')==1), 0).when(col('sexual_orientation_missing')==1, None).otherwise(1))

In [0]:
df_master = df_master.withColumn('baby_phenotypic_sex_label', when((col('baby_phen_sex') == 1) |( col('baby_phen_sex') == '1'), 'male').when((col('baby_phen_sex') == 2) | (col('baby_phen_sex') == '2'), 'female').when((col('baby_phen_sex') == '9') | (col('baby_phen_sex') == 9), 'indeterminate').otherwise(None)).withColumns(
    {
        'baby_male': when(col('baby_phenotypic_sex_label') == 'male', 1).when(col('baby_phenotypic_sex_label').isNull(), None).otherwise(0),
        'baby_female': when(col('baby_phenotypic_sex_label') == 'female', 1).when(col('baby_phenotypic_sex_label').isNull(), None).otherwise(0),
        'baby_indeterminate_sex': when(col('baby_phenotypic_sex_label') == 'indeterminate', 1).when(col('baby_phenotypic_sex_label').isNull(), None).otherwise(0),
        'baby_missing_sex': when(col('baby_phenotypic_sex_label').isNull(), 1).otherwise(0),
    }
)

processed_features_to_keep.extend(["baby_male", "baby_female", "baby_indeterminate_sex"])

In [0]:
df_master = df_master.withColumns({
    "prior_live_birth": when(col("num_prev_births")>=1, 1).when(col("num_prev_births")==0, 0).otherwise(None),
    "prior_csection": when(col("num_prev_csections")>=1, 1).when(col("num_prev_csections")==0, 0).otherwise(None),
    "prior_24w_loss":  when(col("num_prev_24w_losses")>=1, 1).when(col("num_prev_24w_losses")==0, 0).otherwise(None),
    "prior_stillbirth":  when(col("num_prev_still")>=1, 1).when(col("num_prev_still")==0, 0).otherwise(None),
})

processed_features_to_keep.extend(["prior_live_birth", "prior_csection", "prior_24w_loss", "prior_stillbirth"])

In [0]:
df_master = df_master.withColumns({
    "contacts_attended": when(col("contacts_attended")<1.0, None).when(col("contacts_attended")>40.0, 40.0).otherwise(col("contacts_attended")),
    "contacts_facetoface": when(col("contacts_facetoface")<1.0, None).when(col("contacts_facetoface")>40.0, 40.0).otherwise(col("contacts_facetoface")),
}).withColumn(
    "high_total_contacts", 
    when(col("contacts_attended").isNotNull() & (col("contacts_attended")>=20.0), 1).when(col("contacts_attended").isNotNull() & (col("contacts_attended")<20.0), 0).otherwise(None)
)

df_master = df_master.withColumn(
    "percent_contacts_f2f",
    when(col("contacts_attended").isNotNull() & col("contacts_facetoface").isNotNull(), col("contacts_facetoface")/col("contacts_attended")).otherwise(None)
).withColumn(
    "percent_contacts_f2f", 
    when(col("percent_contacts_f2f")>1.0, None).otherwise(col("percent_contacts_f2f"))
).withColumns({
    "high_f2f_contact_percent": when(col("percent_contacts_f2f").isNotNull() & (col("percent_contacts_f2f")>=0.333), 1).when(col("percent_contacts_f2f").isNotNull() & (col("percent_contacts_f2f")<0.333), 0).otherwise(None)
})

processed_features_to_keep.extend(["contacts_attended", "contacts_facetoface", "high_total_contacts", "percent_contacts_f2f", "high_f2f_contact_percent"])

In [0]:
df_master = df_master.withColumns({
    "birth_monday": when(col("birth_day_of_week").isNotNull() & (col("birth_day_of_week")==1), 1).when(col("birth_day_of_week").isNotNull() & (col("birth_day_of_week")!=1), 0).otherwise(None),
    "birth_tuesday": when(col("birth_day_of_week").isNotNull() & (col("birth_day_of_week")==2), 1).when(col("birth_day_of_week").isNotNull() & (col("birth_day_of_week")!=2), 0).otherwise(None),
    "birth_wednesday": when(col("birth_day_of_week").isNotNull() & (col("birth_day_of_week")==3), 1).when(col("birth_day_of_week").isNotNull() & (col("birth_day_of_week")!=3), 0).otherwise(None),
    "birth_thursday": when(col("birth_day_of_week").isNotNull() & (col("birth_day_of_week")==4), 1).when(col("birth_day_of_week").isNotNull() & (col("birth_day_of_week")!=4), 0).otherwise(None),
    "birth_friday": when(col("birth_day_of_week").isNotNull() & (col("birth_day_of_week")==5), 1).when(col("birth_day_of_week").isNotNull() & (col("birth_day_of_week")!=5), 0).otherwise(None),
    "birth_saturday": when(col("birth_day_of_week").isNotNull() & (col("birth_day_of_week")==6), 1).when(col("birth_day_of_week").isNotNull() & (col("birth_day_of_week")!=6), 0).otherwise(None),
    "birth_sunday": when(col("birth_day_of_week").isNotNull() & (col("birth_day_of_week")==7), 1).when(col("birth_day_of_week").isNotNull() & (col("birth_day_of_week")!=7), 0).otherwise(None),
    "birth_weekday": when(col("birth_day_of_week").isNotNull() & col("birth_day_of_week").isin([1,2,3,4,5]), 1).when(col("birth_day_of_week").isNotNull() & ~col("birth_day_of_week").isin([1,2,3,4,5]), 0).otherwise(None),
    "birth_weekend": when(col("birth_day_of_week").isNotNull() & col("birth_day_of_week").isin([6,7]), 1).when(col("birth_day_of_week").isNotNull() & ~col("birth_day_of_week").isin([6,7]), 0).otherwise(None)
}).withColumns({
    "birth_jan": when(col("birth_month").isNotNull() & (col("birth_month")==1), 1).when(col("birth_month").isNotNull() & (col("birth_month")!=1), 0).otherwise(None),
    "birth_feb": when(col("birth_month").isNotNull() & (col("birth_month")==2), 1).when(col("birth_month").isNotNull() & (col("birth_month")!=2), 0).otherwise(None),
    "birth_mar": when(col("birth_month").isNotNull() & (col("birth_month")==3), 1).when(col("birth_month").isNotNull() & (col("birth_month")!=3), 0).otherwise(None),
    "birth_apr": when(col("birth_month").isNotNull() & (col("birth_month")==4), 1).when(col("birth_month").isNotNull() & (col("birth_month")!=4), 0).otherwise(None),
    "birth_may": when(col("birth_month").isNotNull() & (col("birth_month")==5), 1).when(col("birth_month").isNotNull() & (col("birth_month")!=5), 0).otherwise(None),
    "birth_jun": when(col("birth_month").isNotNull() & (col("birth_month")==6), 1).when(col("birth_month").isNotNull() & (col("birth_month")!=6), 0).otherwise(None),
    "birth_jul": when(col("birth_month").isNotNull() & (col("birth_month")==7), 1).when(col("birth_month").isNotNull() & (col("birth_month")!=7), 0).otherwise(None),
    "birth_aug": when(col("birth_month").isNotNull() & (col("birth_month")==8), 1).when(col("birth_month").isNotNull() & (col("birth_month")!=8), 0).otherwise(None),
    "birth_sep": when(col("birth_month").isNotNull() & (col("birth_month")==9), 1).when(col("birth_month").isNotNull() & (col("birth_month")!=9), 0).otherwise(None),
    "birth_oct": when(col("birth_month").isNotNull() & (col("birth_month")==10), 1).when(col("birth_month").isNotNull() & (col("birth_month")!=10), 0).otherwise(None),
    "birth_nov": when(col("birth_month").isNotNull() & (col("birth_month")==11), 1).when(col("birth_month").isNotNull() & (col("birth_month")!=11), 0).otherwise(None),
    "birth_dec": when(col("birth_month").isNotNull() & (col("birth_month")==12), 1).when(col("birth_month").isNotNull() & (col("birth_month")!=12), 0).otherwise(None),
    "birth_winter": when(col("birth_month").isNotNull() & col("birth_month").isin([1,2,3]), 1).when(col("birth_month").isNotNull() & ~col("birth_month").isin([1,2,3]), 0).otherwise(None),
    "birth_spring": when(col("birth_month").isNotNull() & col("birth_month").isin([4,5,6]), 1).when(col("birth_month").isNotNull() & ~col("birth_month").isin([4,5,6]), 0).otherwise(None),
    "birth_summer": when(col("birth_month").isNotNull() & col("birth_month").isin([7,8,9]), 1).when(col("birth_month").isNotNull() & ~col("birth_month").isin([7,8,9]), 0).otherwise(None),
    "birth_autumn": when(col("birth_month").isNotNull() & col("birth_month").isin([10,11,12]), 1).when(col("birth_month").isNotNull() & ~col("birth_month").isin([10,11,12]), 0).otherwise(None)
}).withColumns({
    "birth_am": when(col("birth_am_or_pm").isNotNull() & (col("birth_am_or_pm")==1), 1).when(col("birth_am_or_pm").isNotNull() & (col("birth_am_or_pm")==2), 0).otherwise(None),
    "birth_pm": when(col("birth_am_or_pm").isNotNull() & (col("birth_am_or_pm")==2), 1).when(col("birth_am_or_pm").isNotNull() & (col("birth_am_or_pm")==1), 0).otherwise(None)
}).withColumns({
    "birth_2018": when(col("birth_year").isNotNull() & (col("birth_year") == 2018), 1).when(col("birth_year").isNotNull() & (col("birth_year") != 2018), 0).otherwise(None),
    "birth_2019": when(col("birth_year").isNotNull() & (col("birth_year") == 2019), 1).when(col("birth_year").isNotNull() & (col("birth_year") != 2019), 0).otherwise(None),
    "birth_2020": when(col("birth_year").isNotNull() & (col("birth_year") == 2020), 1).when(col("birth_year").isNotNull() & (col("birth_year") != 2020), 0).otherwise(None),
    "birth_2021": when(col("birth_year").isNotNull() & (col("birth_year") == 2021), 1).when(col("birth_year").isNotNull() & (col("birth_year") != 2021), 0).otherwise(None),
    "birth_2022": when(col("birth_year").isNotNull() & (col("birth_year") == 2022), 1).when(col("birth_year").isNotNull() & (col("birth_year") != 2022), 0).otherwise(None),
    "birth_2023": when(col("birth_year").isNotNull() & (col("birth_year") == 2023), 1).when(col("birth_year").isNotNull() & (col("birth_year") != 2023), 0).otherwise(None),
    "birth_2024": when(col("birth_year").isNotNull() & (col("birth_year") == 2024), 1).when(col("birth_year").isNotNull() & (col("birth_year") != 2024), 0).otherwise(None),
    "birth_2025": when(col("birth_year").isNotNull() & (col("birth_year") == 2025), 1).when(col("birth_year").isNotNull() & (col("birth_year") != 2025), 0).otherwise(None),
    
})

processed_features_to_keep.extend([
    "birth_day_of_week",
    "birth_monday", "birth_tuesday", "birth_wednesday", "birth_thursday", "birth_friday", "birth_saturday", "birth_sunday",
    "birth_weekday", "birth_weekend",
    
    "birth_month",
    "birth_jan", "birth_feb", "birth_mar", "birth_apr", "birth_may", "birth_jun", 
    "birth_jul", "birth_aug", "birth_sep", "birth_oct", "birth_nov", "birth_dec",
    "birth_winter", "birth_spring", "birth_summer", "birth_autumn",

    "birth_am_or_pm",
    "birth_am", "birth_pm",

    "birth_year",
    "birth_2018", "birth_2019", "birth_2020", "birth_2021", "birth_2022", "birth_2023", "birth_2024", "birth_2025"
])

In [0]:
df_master = df_master.withColumn(
    "days_to_first_appointment", datediff(col("antenatal_appt_date"), col("last_period_date"))
)
df_master = df_master.withColumn(
    "is_mother_late", when(col("days_to_first_appointment") > 90, 1).when(col("days_to_first_appointment").isNull(), None).otherwise(0)
)
df_master = df_master.withColumn(
    "last_period_date", when(col("last_period_date").isNull(), F.date_add(col("antenatal_appt_date"), -16*7)).otherwise(col("last_period_date"))
)

processed_features_to_keep.extend(["days_to_first_appointment", "is_mother_late"])

In [0]:
df_master_reduced = df_master.select(processed_features_to_keep)

In [0]:
print(f"Number of rows: {'{:,}'.format(df_master_reduced.count())}. Number of columns: {'{:,}'.format(len(df_master_reduced.columns))}")

In [0]:
write_table_name = "msds_processed_cleaned_vars_stage"

df_master_reduced.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{write_table_name}")